<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue is designed as decision-support for content review, not as an automatic publishing system.

I prioritize content using observed signals from the validated model and the earlier baseline. The main reason codes are:

- `STALE_HIGH_VOLUME` — older content that still receives meaningful impressions. These pages may be worth reviewing first because they have existing visibility.
- `THIN_VISIBLE` — relatively short content that still earns impressions. The research paper observed that growing pages were longer on average, so these pages may be candidates for useful expansion rather than padding.
- `POSITION_OPPORTUNITY` — content with meaningful impressions but weaker average position. These pages may deserve review for search-intent alignment, completeness, and internal linking.
- `REFRESH_REVIEW` — older content that has not been updated recently. The research paper observed an age-related performance pattern and recommends reviewing mature pages before decline becomes difficult to reverse.
- `MONITOR` — pages without a strong observed signal for immediate action.

The ranking is intended to help an editor decide what to inspect first. A higher-ranked item is not proof that updating it will improve performance.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Clients:", df["client_id"].nunique())

# Create the same target used in ML-09
df["target_down"] = (
    df["trend_direction"] == "down"
).astype(int)

# Observed rule-based signals
staleness_score_map = {
    "0-30": 0,
    "31-90": 1,
    "91-180": 2,
    "181+": 3
}

volume_score_map = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3
}

df["staleness_score"] = (
    df["freshness_tier"].map(staleness_score_map)
)

df["volume_score"] = (
    df["impression_tier"].map(volume_score_map)
)

# Action score based on observed signals
df["action_score"] = (
    df["staleness_score"].fillna(0)
    + df["volume_score"].fillna(0)
)

# Reason codes
conditions = [
    (
        (df["staleness_score"] >= 2)
        & (df["volume_score"] >= 2)
    ),
    (
        (df["word_count"] < 2000)
        & (df["impressions_90d"] > 0)
    ),
    (
        (df["impressions_90d"] > 0)
        & (df["avg_position"] > 10)
    ),
    (
        (df["staleness_score"] >= 2)
    )
]

choices = [
    "STALE_HIGH_VOLUME",
    "THIN_VISIBLE",
    "POSITION_OPPORTUNITY",
    "REFRESH_REVIEW"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="MONITOR"
)

# Human-readable action
action_map = {
    "STALE_HIGH_VOLUME":
        "Review and consider refreshing while preserving useful existing content.",
    "THIN_VISIBLE":
        "Review for useful expansion, missing subtopics, examples, and definitions.",
    "POSITION_OPPORTUNITY":
        "Review search intent, completeness, internal links, and page structure.",
    "REFRESH_REVIEW":
        "Review freshness, outdated information, and whether a refresh is warranted.",
    "MONITOR":
        "Monitor performance before taking action."
}

df["recommended_action"] = (
    df["reason_code"].map(action_map)
)

# Rank higher-priority actions first
reason_priority = {
    "STALE_HIGH_VOLUME": 4,
    "THIN_VISIBLE": 3,
    "POSITION_OPPORTUNITY": 2,
    "REFRESH_REVIEW": 1,
    "MONITOR": 0
}

df["priority"] = (
    df["reason_code"].map(reason_priority)
)

queue = (
    df.sort_values(
        ["priority", "action_score", "impressions_90d"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "reason_code",
            "recommended_action",
            "action_score",
            "impressions_90d",
            "word_count",
            "avg_position",
            "freshness_tier"
        ]
    ].head(20)
)

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

Shape: (30000, 44)
Clients: 32


,rank,reason_code,recommended_action,action_score,impressions_90d,word_count,avg_position,freshness_tier
0,1,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,6,61678,5125.0,19.7,181+
1,2,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,6,59472,2591.0,24.8,181+
2,3,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,517715,NaN,4.2,91-180
3,4,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,443434,7676.0,27.9,91-180
4,5,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,347399,NaN,4.2,91-180
5,6,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,309910,2761.0,5.6,91-180
6,7,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,309192,NaN,2.0,91-180
7,8,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,295097,NaN,7.3,91-180
8,9,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,286608,6901.0,26.2,91-180
9,10,STALE_HIGH_VOLUME,Review and consider refreshing while preservin...,5,233561,4610.0,26.2,91-180



Reason-code counts:
reason_code
POSITION_OPPORTUNITY    11951
MONITOR                  8590
THIN_VISIBLE             4520
STALE_HIGH_VOLUME        3524
REFRESH_REVIEW           1415
Name: count, dtype: int64


## 2. Intended use and limits

### Intended use

This playbook is intended for editors or SEO/content teams who need to decide which content to review first. It converts observed model and content signals into a ranked decision-support queue.

The queue can help with prioritization, such as identifying older pages with existing visibility, thin pages that still receive impressions, and pages with search-position opportunities.

The ranking is not a production decision system. It does not predict with certainty that a page will decline or that a recommended action will improve performance.

### Limits

The observed relationships are directional and should not be interpreted as causal effects. The research findings are based on observational portfolio data, and the model was evaluated on grouped client data.

The queue should therefore be used to prioritize human review rather than automatically publish, rewrite, delete, redirect, or change content.

The recommendations may also become less reliable if the underlying content mix, search environment, client portfolio, or feature distributions change substantially.

### Cost and value thinking

Higher-priority actions should be reviewed first when they combine meaningful existing visibility with a plausible content-maintenance opportunity. This can make human review more efficient than treating every page equally.

Before spending significant editing resources, reviewers should consider expected value, page importance, existing traffic, effort required, and whether the recommended change has a clear content rationale.

In [3]:
print("Intended users: editors and SEO/content teams")
print("Purpose: content prioritization and decision support")
print("Queue size:", len(queue))

print("\nPriority distribution:")
display(
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

print(
    "\nThe queue is intended for human-reviewed prioritization, "
    "not automated content changes."
)

Intended users: editors and SEO/content teams
Purpose: content prioritization and decision support
Queue size: 30000

Priority distribution:


,reason_code,count
0,POSITION_OPPORTUNITY,11951
1,MONITOR,8590
2,THIN_VISIBLE,4520
3,STALE_HIGH_VOLUME,3524
4,REFRESH_REVIEW,1415



The queue is intended for human-reviewed prioritization, not automated content changes.


## 3. Human review + the no-go list

The ranked queue is decision-support only. A human editor or SEO/content specialist must review an item before taking action.

### Human review rules

Before acting on a recommendation, the reviewer should:

1. Check the current content and search intent.
2. Confirm that the recommended action is appropriate for the topic.
3. Review recent performance and content freshness.
4. Check whether the page has important business, legal, or brand context.
5. Consider whether refreshing, monitoring, or leaving the page unchanged is the best choice.
6. Treat the reason code as a prioritization signal, not as proof that an action will improve performance.

### No-go list

The system should not automatically:

- Publish or rewrite content.
- Delete or redirect pages.
- Change important business, legal, medical, or brand claims.
- Make irreversible SEO changes.
- Guarantee that a page will decline or improve.
- Treat the ranking score as a final editorial decision.

Human judgment is required before any content change is made.

In [4]:
print("Human review is required before any recommended action.")
print("No-go cases: automated publishing, deletion, redirects, irreversible SEO changes, and guaranteed performance claims.")

Human review is required before any recommended action.
No-go cases: automated publishing, deletion, redirects, irreversible SEO changes, and guaranteed performance claims.


## 4. Monitoring / retrain triggers

The recommendations should be monitored because content performance and search behavior can change over time.

I would review the recommendation queue periodically and check whether the observed outcomes still align with the reason codes and intended actions.

### Monitoring triggers

- Recheck recommendation performance when the measured F1 score drops meaningfully from the validated result.
- Review the queue if the distribution of reason codes changes substantially.
- Reassess the feature signals if search visibility, content age, or traffic patterns change.
- Review recommendations when the content environment or search behavior changes enough that the current signals may no longer represent the same decision problem.

### Retrain trigger

Retraining should be considered when new labeled data becomes available and the measured model performance shows a sustained decline under the same validation design used for the audit.

These triggers are intended as monitoring and decision-support rules, not as automatic production actions.

In [5]:
# Monitoring / retrain trigger checks

monitoring_triggers = {
    "performance_check": "Review if measured F1 shows a sustained decline from the validated result.",
    "reason_code_shift": "Review if the distribution of reason codes changes substantially.",
    "feature_shift": "Review if key feature patterns such as visibility, traffic, or content age change substantially.",
    "environment_change": "Review if search behavior or the content environment changes enough to affect the current signals.",
    "retrain_trigger": "Consider retraining when new labeled data is available and performance declines under the same validation design."
}

for trigger, description in monitoring_triggers.items():
    print(f"{trigger}: {description}")

print("\nMonitoring checks defined:", len(monitoring_triggers))
print("These are review triggers, not automatic production actions.")

performance_check: Review if measured F1 shows a sustained decline from the validated result.
reason_code_shift: Review if the distribution of reason codes changes substantially.
feature_shift: Review if key feature patterns such as visibility, traffic, or content age change substantially.
environment_change: Review if search behavior or the content environment changes enough to affect the current signals.
retrain_trigger: Consider retraining when new labeled data is available and performance declines under the same validation design.

Monitoring checks defined: 5
These are review triggers, not automatic production actions.


## 5. Exports for the paper

The ranked action queue is exported for reuse in the research paper. The queue is regenerated by the notebook rather than treated as a committed data file.

The export contains the ranked recommendations, reason codes, recommended actions, and supporting signals used for prioritization.

The exported queue is intended for analysis and human-reviewed decision support, not automated production changes.

In [7]:
from pathlib import Path

# Create output directory
OUTPUT_DIR = Path("/content/work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue
queue_path = OUTPUT_DIR / "content_action_queue.csv"
queue.to_csv(queue_path, index=False)

print("Exported queue:", queue_path)
print("Rows exported:", len(queue))
print("Columns exported:", len(queue.columns))

Exported queue: /content/work/outputs/content_action_queue.csv
Rows exported: 30000
Columns exported: 52


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.